# A/B Testing
## Comparing Two Samples

## The Data

`baby.csv` has one row per baby, from a random sample of mothers and newborns in a large hospital system. We'll use two columns: `Birth Weight` (in ounces) and `Maternal Smoker` (`TRUE` if the mother smoked during pregnancy).

In [ ]:
library(tidyverse)
library(infer)
options(repr.matrix.max.rows = 20)

births <- read_csv("data/baby.csv", show_col_types = FALSE)
births

## The Groups

How many mothers are in each group?

In [ ]:
smoking_and_birthweight <- births |>
  select(`Maternal Smoker`, `Birth Weight`)

smoking_and_birthweight |>
  count(`Maternal Smoker`)

The groups are different sizes, so we plot densities instead of counts. `position = "identity"` overlays the two histograms instead of stacking them.

In [ ]:
smoking_and_birthweight |>
  ggplot(aes(x = `Birth Weight`, y = after_stat(density), fill = `Maternal Smoker`)) +
  geom_histogram(position = "identity", alpha = 0.5, bins = 15, color = "white") +
  labs(title = "Birth weights by maternal smoking",
       x = "Birth weight (ounces)", y = "Density") +
  theme_minimal(base_size = 16)

## Hypotheses

**Null hypothesis:** In the population, birth weights have the same distribution for babies of smokers and non-smokers. The difference in our sample is due to chance.

**Alternative hypothesis:** In the population, babies of mothers who smoked weigh less on average than babies of non-smokers.

## Test Statistic

Our statistic is the difference in average birth weight: **smokers' average minus non-smokers' average**. Negative values favor the alternative.

In [ ]:
means_table <- smoking_and_birthweight |>
  group_by(`Maternal Smoker`) |>
  summarize(average = mean(`Birth Weight`))
means_table

`FALSE` sorts before `TRUE`, so the non-smokers are in row 1 and the smokers in row 2.

In [ ]:
means <- means_table |> pull(average)
observed_difference <- means[2] - means[1]
observed_difference

We'll compute this difference many times, so we put the steps in a function. It takes a table, the name of the numerical column, and the name of the group column. `.data[[...]]` tells dplyr to find the column whose name is stored in the variable.

In [ ]:
difference_of_means <- function(table, numerical_label, group_label) {
  means <- table |>
    group_by(.data[[group_label]]) |>
    summarize(average = mean(.data[[numerical_label]])) |>
    pull(average)
  means[2] - means[1]
}

difference_of_means(births, "Birth Weight", "Maternal Smoker")

## Shuffling the Labels

If the null hypothesis is true, the smoker label tells us nothing about birth weight. So we can shuffle the labels, keep each birth weight where it is, and recompute the difference.

`sample()` with no size shuffles a vector: it uses every element once, without replacement.

In [ ]:
set.seed(42)
original_and_shuffled <- smoking_and_birthweight |>
  mutate(`Shuffled Label` = sample(`Maternal Smoker`))
original_and_shuffled

Shuffling without replacement keeps the group sizes the same:

In [ ]:
original_and_shuffled |>
  count(`Shuffled Label`)

In [ ]:
difference_of_means(original_and_shuffled, "Birth Weight", "Shuffled Label")

## One Simulated Difference

This function does one shuffle and returns the difference in means. Run it a few times.

In [ ]:
one_simulated_difference <- function(table, numerical_label, group_label) {
  shuffled_table <- table |>
    mutate(`Shuffled Label` = sample(.data[[group_label]]))
  difference_of_means(shuffled_table, numerical_label, "Shuffled Label")
}

one_simulated_difference(births, "Birth Weight", "Maternal Smoker")

## Repeat Many Times

A `for` loop collects 1,000 simulated differences. This takes a few seconds.

In [ ]:
set.seed(42)
differences <- c()

repetitions <- 1000
for (i in 1:repetitions) {
  new_difference <- one_simulated_difference(births, "Birth Weight", "Maternal Smoker")
  differences <- c(differences, new_difference)
}

Where should this distribution be centered? Where does our observed difference (the red line) fall?

In [ ]:
tibble(difference = differences) |>
  ggplot(aes(x = difference)) +
  geom_histogram(bins = 20, fill = "steelblue", color = "white") +
  geom_vline(xintercept = observed_difference, color = "red", linewidth = 1) +
  labs(title = "Prediction under the null hypothesis",
       x = "Difference in average birth weight (smokers - non-smokers)", y = "Count") +
  theme_minimal(base_size = 16)

## p-value

Low values favor the alternative, so the p-value is the share of simulated differences at or below the observed difference.

In [ ]:
sum(differences <= observed_difference) / repetitions

## The Same Test with `infer`

The `infer` pipeline does the same steps:

- `specify()` names the response and the group variable as `response ~ group`
- `hypothesize(null = "independence")` says the response doesn't depend on the group
- `generate(type = "permute")` shuffles the group labels, once per repetition
- `calculate(stat = "diff in means")` computes the statistic for each shuffle. `order = c("TRUE", "FALSE")` subtracts in the same order as before: smokers minus non-smokers.

`infer` needs the group column as text, not `TRUE`/`FALSE`, so we convert it first.

In [ ]:
births_chr <- births |>
  mutate(`Maternal Smoker` = as.character(`Maternal Smoker`))

observed_stat <- births_chr |>
  specify(`Birth Weight` ~ `Maternal Smoker`) |>
  calculate(stat = "diff in means", order = c("TRUE", "FALSE"))
observed_stat

In [ ]:
set.seed(42)
null_dist <- births_chr |>
  specify(`Birth Weight` ~ `Maternal Smoker`) |>
  hypothesize(null = "independence") |>
  generate(reps = 2500, type = "permute") |>
  calculate(stat = "diff in means", order = c("TRUE", "FALSE"))

null_dist |>
  visualize() +
  shade_p_value(obs_stat = observed_stat, direction = "less")

In [ ]:
null_dist |>
  get_p_value(obs_stat = observed_stat, direction = "less")

## Conclusion

None of the shuffles came close to the observed difference, so the p-value is about 0. (`infer` warns that a p-value of 0 is an approximation based on the number of repetitions.) The data support the alternative: babies of mothers who smoked weigh less on average.

**Can we say that smoking caused the lower birth weights?**

## Try it out!

Do smokers and non-smokers differ in age?

- Predict first: which group is younger on average?
- Use `difference_of_means()` to find the observed difference in `Maternal Age` (smokers minus non-smokers).
- Run the permutation test with `infer`. Which `direction` matches the alternative "smokers are younger on average"?
- What do you conclude at the 5% cutoff? What does this tell you about whether smoking caused the lower birth weights?

In [ ]:
# Your code here